## End-to-End Machine Learning Pipeline
This notebook covers the entire workflow of a machine learning project, from loading and exploring the data to deploying the model.
It includes data splitting, exploratory data analysis, preprocessing, training, evaluation, hyperparameter tuning, and final evaluation.

### Importing Libraries
The necessary libraries for this project are imported below.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
np.random.seed(42)

### Step 1: Load and Explore Data
The dataset used in this example is the breast cancer dataset from sklearn.

In [ ]:
from sklearn.datasets import load_breast_cancer
data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target, name='target')

print("\n=== STEP 1: LOAD AND EXPLORE ===")
print(f"Dataset shape: {X.shape}")
print(f"Features: {X.shape[1]}")
print(f"Samples: {X.shape[0]}")
print(f"\nFirst few rows:\n")
print(X.head())

The output of the above code shows the shape of the dataset, the number of features, and the number of samples.
One common mistake is not checking the shape of the dataset before proceeding.
A quick tip is to always use the head() function to view the first few rows of the dataset.

### Data Quality Check
The data quality check is performed to identify any missing values or inconsistent data types.

In [ ]:
print(f"\nData quality:\n")
print(f"  Missing values: {X.isnull().sum().sum()}")
print(f"  Data types: {X.dtypes.unique()}")
print(f"  Feature ranges:\n")
for col in X.columns[:3]:  # Show first 3
    print(f"    {col}: [{X[col].min():.2f}, {X[col].max():.2f}]")

The output of the above code shows the number of missing values, the data types of the features, and the range of the first three features.
One common mistake is not checking for missing values before proceeding.
A quick tip is to always use the isnull() function to check for missing values.

### Target Distribution
The target distribution is checked to identify the number of samples in each class.

In [ ]:
print(f"\nTarget distribution:\n")
print(f"  Class 0 (Malignant): {(y==0).sum()} samples")
print(f"  Class 1 (Benign): {(y==1).sum()} samples")
print(f"  Balance ratio: {(y==1).sum() / (y==0).sum():.2f}")

The output of the above code shows the number of samples in each class and the balance ratio.
One common mistake is not checking the target distribution before proceeding.
A quick tip is to always use the value_counts() function to check the target distribution.

### Step 2: Data Splitting
The dataset is split into training and testing sets using the train_test_split function.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train set size: {len(X_train)} ({len(X_train)/len(X)*100:.1f}%)")
print(f"Test set size: {len(X_test)} ({len(X_test)/len(X)*100:.1f}%)")
print(f"\nTrain set class distribution:\n")
print(f"  Class 0: {(y_train==0).sum()}")
print(f"  Class 1: {(y_train==1).sum()}")
print(f"Test set class distribution:\n")
print(f"  Class 0: {(y_test==0).sum()}")
print(f"  Class 1: {(y_test==1).sum()}")

The output of the above code shows the size of the training and testing sets and the class distribution.
One common mistake is not using stratified splitting.
A quick tip is to always use the stratify parameter to maintain the class distribution in the training and testing sets.

### Step 3: Exploratory Data Analysis (EDA)
The exploratory data analysis is performed to understand the patterns and correlations in the data.

In [ ]:
print(f"Feature statistics (training set):\n")
print(f"  Mean feature value: {X_train.mean().mean():.4f}")
print(f"  Std feature value: {X_train.std().mean():.4f}")
print(f"  Feature ranges vary widely (0-1000+)")

corr_matrix = X_train.corr()
high_corr = np.where(np.abs(corr_matrix) > 0.95)
high_corr_pairs = [(X.columns[i], X.columns[j]) for i, j in zip(*high_corr) if i < j]
print(f"\nHighly correlated feature pairs (>0.95):\n")
for i, (feat1, feat2) in enumerate(high_corr_pairs[:3]):
    corr_val = corr_matrix.loc[feat1, feat2]
    print(f"  {feat1} <-> {feat2}: {corr_val:.4f}")

skewness = X_train.skew()
skewed_features = skewness[np.abs(skewness) > 1].sort_values(ascending=False)
print(f"\nHighly skewed features (|skewness| > 1):\n")
for feat, skew_val in skewed_features.head(3).items():
    print(f"  {feat}: {skew_val:.4f}")

The output of the above code shows the feature statistics, highly correlated feature pairs, and highly skewed features.
One common mistake is not checking for correlations and skewness.
A quick tip is to always use the corr() function to check for correlations and the skew() function to check for skewness.

### Step 4: Preprocessing & Feature Scaling
The preprocessing and feature scaling is performed to prepare the data for modeling.

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"After StandardScaler:\n")
print(f"  Training set mean: {X_train_scaled.mean():.4f}")
print(f"  Training set std: {X_train_scaled.std():.4f}")
print(f"  Test set mean: {X_test_scaled.mean():.4f}")
print(f"  Test set std: {X_test_scaled.std():.4f}")

The output of the above code shows the mean and standard deviation of the training and testing sets after scaling.
One common mistake is not scaling the data.
A quick tip is to always use the StandardScaler to scale the data.

### Step 5: Train Baseline Model
The baseline model is trained to provide a baseline for comparison with other models.

In [ ]:
baseline = LogisticRegression(max_iter=1000, random_state=42)
baseline.fit(X_train_scaled, y_train)

y_train_pred = baseline.predict(X_train_scaled)
y_test_pred = baseline.predict(X_test_scaled)

print(f"Logistic Regression performance:\n")
print(f"  Train accuracy: {accuracy_score(y_train, y_train_pred):.4f}")
print(f"  Test accuracy: {accuracy_score(y_test, y_test_pred):.4f}")
print(f"  Train/Test gap: {accuracy_score(y_train, y_train_pred) - accuracy_score(y_test, y_test_pred):.4f}")

The output of the above code shows the performance of the baseline model.
One common mistake is not evaluating the model on the test set.
A quick tip is to always use the accuracy_score function to evaluate the model.

### Step 6: Detailed Evaluation
The detailed evaluation is performed to get a better understanding of the model's performance.

In [ ]:
y_pred_baseline = baseline.predict(X_test_scaled)
accuracy = accuracy_score(y_test, y_pred_baseline)
precision = precision_score(y_test, y_pred_baseline)
recall = recall_score(y_test, y_pred_baseline)
f1 = f1_score(y_test, y_pred_baseline)

print(f"Binary classification metrics (test set):\n")
print(f"  Accuracy:  {accuracy:.4f} (correct predictions / total)")
print(f"  Precision: {precision:.4f} (true positives / predicted positives)")
print(f"  Recall:    {recall:.4f} (true positives / actual positives)")
print(f"  F1 Score:  {f1:.4f} (harmonic mean of precision and recall)")

The output of the above code shows the detailed evaluation metrics of the model.
One common mistake is not using the correct evaluation metrics.
A quick tip is to always use the accuracy_score, precision_score, recall_score, and f1_score functions to evaluate the model.

### Step 7: Try Advanced Model
The advanced model is trained to compare with the baseline model.

In [ ]:
rf = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
rf.fit(X_train, y_train)  # Note: RF doesn't need scaling

y_rf_train_pred = rf.predict(X_train)
y_rf_test_pred = rf.predict(X_test)

print(f"Random Forest performance:\n")
print(f"  Train accuracy: {accuracy_score(y_train, y_rf_train_pred):.4f}")
print(f"  Test accuracy: {accuracy_score(y_test, y_rf_test_pred):.4f}")
print(f"  Train/Test gap: {accuracy_score(y_train, y_rf_train_pred) - accuracy_score(y_test, y_rf_test_pred):.4f}")

The output of the above code shows the performance of the advanced model.
One common mistake is not comparing the performance of different models.
A quick tip is to always use the accuracy_score function to compare the performance of different models.

### Step 8: Hyperparameter Tuning
The hyperparameter tuning is performed to find the best hyperparameters for the model.

In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid_rf = {
    'max_depth': [5, 10, 15],
    'min_samples_leaf': [1, 2, 4],
    'n_estimators': [50, 100, 200]
}

grid_search = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid_rf,
    cv=5,
    scoring='f1',  # Optimize for F1 (balanced metric)
    n_jobs=-1  # Use all CPU cores
)

grid_search.fit(X_train, y_train)

print(f"Best parameters: {grid_search.best_params_}")
print(f"Best CV score: {grid_search.best_score_:.4f}")

The output of the above code shows the best hyperparameters and the best cross-validation score.
One common mistake is not using cross-validation to evaluate the model.
A quick tip is to always use the GridSearchCV function to perform hyperparameter tuning.

### Step 9: Final Evaluation on Holdout Test Set
The final evaluation is performed on the holdout test set to get a final estimate of the model's performance.

In [ ]:
best_rf = grid_search.best_estimator_
best_test_acc = best_rf.score(X_test, y_test)

print(f"Best model test accuracy: {best_test_acc:.4f}")

The output of the above code shows the final estimate of the model's performance on the holdout test set.
One common mistake is not evaluating the model on the holdout test set.
A quick tip is to always use the score function to evaluate the model on the holdout test set.

### Step 10: Save for Deployment
The model is saved for deployment.

In [ ]:
import pickle
model_path = "c:\\Users\\SOURAV\\Desktop\\Pybasics\\15_ml_workflow\\best_model.pkl"
scaler_path = "c:\\Users\\SOURAV\\Desktop\\Pybasics\\15_ml_workflow\\scaler.pkl"

pickle.dump(best_rf, open(model_path, 'wb'))
pickle.dump(scaler, open(scaler_path, 'wb'))

print(f"Model saved to: {model_path}")
print(f"Scaler saved to: {scaler_path}")

The output of the above code shows the path where the model and scaler are saved.
One common mistake is not saving the model and scaler.
A quick tip is to always use the pickle function to save the model and scaler.

## Key Takeaways
* The machine learning pipeline consists of loading and exploring the data, splitting the data, exploratory data analysis, preprocessing, training, evaluation, hyperparameter tuning, and final evaluation.
* Always split the data before any preprocessing to prevent data leakage.
* Use cross-validation to evaluate the model and perform hyperparameter tuning.
* Compare the performance of different models and use the best model for deployment.
* Save the model and scaler for deployment.